# Depression Detection – Audio Pipeline (ViT)
**Dataset:** DAIC-WOZ | **Model:** ViT-Base Patch16 | **Input:** Mel-Spectrograms

Authors: Ayush Negi (22BLC1259), Ayush Arora (22BLC1218), Karan Ghuwalewala (22BLC1201)

VIT Chennai – School of Electronics Engineering, Nov 2025

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install timm librosa noisereduce pydub soundfile -q
!apt-get install -y ffmpeg -q

## 1. Imports & Config

In [ ]:
import os, re, gc, copy, time, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display, soundfile as sf
import noisereduce as nr
from pydub import AudioSegment
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

BASE_DIR        = '/content/drive/MyDrive/DDS'
AUDIO_DIR       = os.path.join(BASE_DIR, 'AUDIO')
TRANSCRIPT_DIR  = os.path.join(BASE_DIR, 'DATAS')
CLEANED_AUDIO   = os.path.join(BASE_DIR, 'Cleaned_Audio')
DENOISED_AUDIO  = os.path.join(BASE_DIR, 'Denoised_Audio')
SPECTROGRAM_DIR = os.path.join(BASE_DIR, 'Spectrograms_Optimized')
RESULTS_DIR     = os.path.join(BASE_DIR, 'ViT_Results')
CSV_DIR         = os.path.join(BASE_DIR, 'csvs')
for d in [CLEANED_AUDIO, DENOISED_AUDIO, SPECTROGRAM_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE, BATCH_SIZE, EPOCHS = 224, 8, 25
LR, PATIENCE, SEED = 1e-4, 6, 42
MODEL_NAME = 'vit_base_patch16_224'
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, 'best_vit.pth')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {device}')

## 2. Extract Participant Audio Segments

In [ ]:
def extract_id(fname):
    m = re.match(r'(\d+)', fname); return m.group(1) if m else None

for filename in os.listdir(TRANSCRIPT_DIR):
    if not filename.endswith('.csv'): continue
    tid = extract_id(filename)
    if not tid: continue
    matching = [a for a in os.listdir(AUDIO_DIR) if tid in a and a.endswith('.wav')]
    if not matching: continue
    out_path = os.path.join(CLEANED_AUDIO, f'{tid}_Participant.wav')
    if os.path.exists(out_path): continue
    df = pd.read_csv(os.path.join(TRANSCRIPT_DIR, filename))
    p_df = df[df['speaker'] == 'Participant']
    audio = AudioSegment.from_wav(os.path.join(AUDIO_DIR, matching[0]))
    p_audio = AudioSegment.empty()
    for _, row in p_df.iterrows():
        p_audio += audio[row['start_time']*1000 : row['stop_time']*1000]
    p_audio.export(out_path, format='wav')
    print(f'Saved: {out_path}')
print('Done extracting participant audio.')

## 3. Denoise Audio

In [ ]:
for filename in os.listdir(CLEANED_AUDIO):
    if not filename.endswith('.wav'): continue
    out_path = os.path.join(DENOISED_AUDIO, filename.replace('.wav','_denoised.wav'))
    if os.path.exists(out_path): continue
    y, sr = librosa.load(os.path.join(CLEANED_AUDIO, filename), sr=None)
    noise = y[:int(0.5*sr)]
    reduced = nr.reduce_noise(y=y, y_noise=noise, sr=sr, prop_decrease=0.95)
    sf.write(out_path, reduced, sr)
    print(f'Denoised: {filename}')
print('All files denoised.')

## 4. Generate Mel-Spectrograms

In [ ]:
wav_files = [f for f in os.listdir(DENOISED_AUDIO) if f.endswith('.wav')]
batch_size = 10
for i in range(0, len(wav_files), batch_size):
    batch = wav_files[i:i+batch_size]
    print(f'Batch {i//batch_size+1}/{len(wav_files)//batch_size+1}')
    for filename in batch:
        out_path = os.path.join(SPECTROGRAM_DIR, filename.replace('.wav','_spec.png'))
        if os.path.exists(out_path): continue
        try:
            y, sr = librosa.load(os.path.join(DENOISED_AUDIO, filename), sr=None)
            S  = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
            Sd = librosa.power_to_db(S, ref=np.max)
            plt.figure(figsize=(8,3))
            librosa.display.specshow(Sd, sr=sr, x_axis='time', y_axis='mel', cmap='magma')
            plt.axis('off')
            plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
            plt.close('all')
        except Exception as e:
            print(f'  Skipped {filename}: {e}')
    gc.collect()
print('All spectrograms generated.')

## 5. Load Labels & Build Dataset

In [ ]:
csv_files = glob.glob(os.path.join(CSV_DIR,'*.csv')) + glob.glob(os.path.join(CSV_DIR,'*.xls*'))
dfs = []
for p in csv_files:
    try: dfs.append(pd.read_csv(p))
    except: dfs.append(pd.read_excel(p))
labels_df = pd.concat(dfs)[['Participant_ID','PHQ_Binary']].drop_duplicates().reset_index(drop=True)
labels_df['Participant_ID'] = labels_df['Participant_ID'].astype(str)

spec_files = sorted([f for f in os.listdir(SPECTROGRAM_DIR) if f.lower().endswith('.png')])
files_df   = pd.DataFrame([{'filename':f,'Participant_ID':extract_id(f)} for f in spec_files])
merged     = files_df.merge(labels_df, on='Participant_ID', how='left')
merged     = merged.dropna(subset=['PHQ_Binary']).reset_index(drop=True)
merged['label'] = merged['PHQ_Binary'].astype(int)
print(f'Labeled spectrograms: {len(merged)}')
print(merged['label'].value_counts())

## 6. Vision Transformer Training

In [ ]:
class SpectrogramDataset(Dataset):
    def __init__(self, df, root, transform=None):
        self.df = df.reset_index(drop=True); self.root = root; self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.root, row['filename'])).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row['label']), row['filename']

mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.1,0.1,0.1,0.1)], p=0.5),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

train_df, val_df = train_test_split(merged, test_size=0.2, stratify=merged['label'], random_state=SEED)
train_loader = DataLoader(SpectrogramDataset(train_df, SPECTROGRAM_DIR, train_tf), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SpectrogramDataset(val_df,   SPECTROGRAM_DIR, val_tf),   batch_size=BATCH_SIZE)

model   = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0)
in_ch   = model.num_features
model.head = nn.Sequential(nn.Linear(in_ch,256), nn.GELU(), nn.Dropout(0.3), nn.Linear(256,2))
model   = model.to(device)

labels_all   = merged['label'].values
class_counts = np.bincount(labels_all)
weights      = torch.tensor([len(labels_all)/(2*c) for c in class_counts], dtype=torch.float32).to(device)
criterion    = nn.CrossEntropyLoss(weight=weights)
optimizer    = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler    = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()
best_acc, best_state, no_improve = 0.0, None, 0

for epoch in range(1, EPOCHS+1):
    model.train(); tr_preds, tr_labels = [], []
    for imgs, lbls, _ in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        with autocast(): out = model(imgs); loss = criterion(out, lbls)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr_preds.extend(out.argmax(1).cpu().tolist()); tr_labels.extend(lbls.cpu().tolist())
    model.eval(); vp, vl, vl_sum = [], [], 0.0
    with torch.no_grad():
        for imgs, lbls, _ in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            with autocast(): out = model(imgs); loss = criterion(out, lbls)
            vl_sum += loss.item()*imgs.size(0)
            vp.extend(out.argmax(1).cpu().tolist()); vl.extend(lbls.cpu().tolist())
    val_loss = vl_sum/len(val_loader.dataset)
    val_acc  = accuracy_score(vl, vp); scheduler.step(val_loss)
    print(f'Epoch {epoch:02d}: Train {accuracy_score(tr_labels,tr_preds):.3f} | Val {val_acc:.3f} | Loss {val_loss:.4f}')
    if val_acc > best_acc:
        best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_MODEL_PATH); no_improve = 0
        print(f'  ✅ Saved (Val Acc={best_acc:.3f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE: print('Early stopping.'); break

print(f'Best Val Acc: {best_acc:.3f}')

## 7. Confusion Matrix & ROC Curve

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve, auc as sk_auc

if best_state: model.load_state_dict(best_state)
model.eval(); y_true, y_pred, y_prob = [], [], []
all_loader = DataLoader(SpectrogramDataset(merged, SPECTROGRAM_DIR, val_tf), batch_size=BATCH_SIZE)
with torch.no_grad():
    for imgs, lbls, _ in all_loader:
        imgs = imgs.to(device)
        with autocast(): out = model(imgs)
        prob = torch.softmax(out, dim=-1)[:,1].cpu().numpy()
        y_pred.extend(out.argmax(1).cpu().tolist())
        y_true.extend(lbls.tolist()); y_prob.extend(prob)

cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Dep','Dep'], yticklabels=['Not Dep','Dep'])
axes[0].set_title('Confusion Matrix')
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = sk_auc(fpr, tpr)
axes[1].plot(fpr, tpr, label=f'AUC={roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='gray')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f'Final Accuracy: {accuracy_score(y_true, y_pred):.4f}')